# MNIST CNN — Kaggle Notebook training

Stretch-goal CNN comparison against the MLP results. Same setup as `kaggle_train.ipynb` — see that notebook's intro cell for the full Add Input / Settings walkthrough. Quick recap:
1. From your `mnist-mlp-pipeline` dataset page, **⋮ menu → New notebook** (or reuse an existing one with the dataset already attached — just make sure it has the latest version, published after CNN support was added).
2. **Add Input** → search `digit-recognizer` → add the trophy-icon "Ongoing"/"Getting Started" competition card.
3. **Settings → Accelerator → GPU T4 x2**, **Settings → Environment Preferences → Internet → On**.

Input paths (confirmed from the MLP run): datasets mount at `/kaggle/input/datasets/<owner>/<dataset-slug>/`, competitions at `/kaggle/input/competitions/<competition-slug>/`.

In [ ]:
!pip install -q pytorch-lightning torchmetrics optuna python-dotenv

In [ ]:
import sys

DATASET_DIR = '/kaggle/input/datasets/ivannamor/mnist-mlp-pipeline'
sys.path.insert(0, DATASET_DIR)
print('project code ready')

In [ ]:
from src.config import ExperimentConfig

config = ExperimentConfig.from_yaml(f'{DATASET_DIR}/configs/cnn_baseline.yaml')
config.data.data_dir = '/kaggle/input/competitions/digit-recognizer'
config.trainer.experiment_name = 'cnn_baseline'
config

## Baseline CNN training run (GPU)

In [ ]:
from src.train import train

trainer = train(config)

## Hyperparameter search (Optuna)

Same `search.objective`/`search.build_config` as the MLP run — they dispatch on `config.architecture` (already `'cnn'` here) to sample conv_channels/kernel_size/fc_hidden_dim instead of hidden_dims. No per-trial checkpoints (by design); the next cell retrains the winner for real.

In [ ]:
import optuna
from src.search import objective

study = optuna.create_study(direction='maximize', pruner=optuna.pruners.MedianPruner())
study.optimize(lambda trial: objective(trial, config), n_trials=20)

print('Best value:', study.best_value)
print('Best params:', study.best_params)

## Retrain the winning configuration

In [ ]:
from src.search import build_config

best_config = build_config(study.best_params, config)
best_config.trainer.experiment_name = 'cnn_best_from_search'
best_trainer = train(best_config)

## Final test-set evaluation

Run once. Uses the same stratified test split as the MLP run (same seed, same data), so this is a fair, directly comparable number.

In [ ]:
from pathlib import Path
from src.evaluate import evaluate

checkpoint_path = best_trainer.checkpoint_callback.best_model_path
evaluate(best_config, checkpoint_path, Path('/kaggle/working/evaluation'))

## Download results

Kaggle's `FileLink` 404'd for the MLP run's downloads — this base64 data-URI approach worked instead.

In [ ]:
import base64
from IPython.display import HTML, display

def make_download_link(path):
    filename = path.split('/')[-1]
    with open(path, 'rb') as f:
        data = base64.b64encode(f.read()).decode()
    return HTML(f'<a download="{filename}" href="data:application/octet-stream;base64,{data}">Download {filename}</a>')

display(make_download_link(checkpoint_path))
display(make_download_link('/kaggle/working/evaluation/confusion_matrix.png'))
display(make_download_link('/kaggle/working/evaluation/misclassified_examples.png'))